In [ ]:
# 1. ?????


import os
import copy
import numpy as np
import pandas as pd
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MinMaxScaler

import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

In [ ]:
import torch
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. 4~8월 cargo CSV 로딩

file_paths = [
    r"C:\Users\HP\Desktop\prE\04cargo.csv",
    r"C:\Users\HP\Desktop\prE\05cargo.csv",
    r"C:\Users\HP\Desktop\prE\06cargo.csv",
    r"C:\Users\HP\Desktop\prE\07cargo.csv",
    r"C:\Users\HP\Desktop\prE\08cargo.csv",
]

df_list = []

for path in file_paths:
    temp = pd.read_csv(path)
    temp["source_file"] = os.path.basename(path)
    df_list.append(temp)

df_raw = pd.concat(df_list, ignore_index=True)

print(df_raw.shape)
print(df_raw.columns)
df_raw.head()

In [ ]:
# 3. 컬럼 정리 및 기본 전처리

df = df_raw[["MMSI", "Date", "Lat", "Long", "SOG", "COG", "Heading", "source_file"]].copy()

df = df.rename(columns={
    "Date": "timestamp",
    "Lat": "lat",
    "Long": "lon",
    "SOG": "sog",
    "COG": "cog",
    "Heading": "heading"
})

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

for col in ["lat", "lon", "sog", "cog", "heading"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["MMSI", "timestamp", "lat", "lon", "sog", "cog", "heading"])

df = df[
    (df["lat"].between(-90, 90)) &
    (df["lon"].between(-180, 180)) &
    (df["sog"] >= 0) &
    (df["cog"].between(0, 360)) &
    (df["heading"].between(0, 360))
].copy()

df = df.sort_values(["MMSI", "timestamp"]).reset_index(drop=True)

df["month"] = df["timestamp"].dt.month

print(df.shape)
print(df["month"].value_counts().sort_index())
df.head()

In [ ]:
# 4. 이동 선박 / 정지 선박 기준

MOVING_THRESHOLD = 0.5

moving_df = df[df["sog"] > MOVING_THRESHOLD].copy()
stationary_df = df[df["sog"] <= MOVING_THRESHOLD].copy()

print("전체 데이터:", len(df))
print("이동 선박 데이터:", len(moving_df))
print("정지/저속 선박 데이터:", len(stationary_df))

In [ ]:
# 5. Grid 설정
# Interaction Graph 학습용: 0.5 NM
# 최종 혼잡도 시각화용: 1.0 NM

TRAIN_GRID_NM = 0.1
VIS_GRID_NM = 0.1

BASE_LAT_FOR_LON_SCALE = 35.0

TRAIN_LAT_GRID_SIZE = (TRAIN_GRID_NM * 1.852) / 111
TRAIN_LON_GRID_SIZE = (TRAIN_GRID_NM * 1.852) / (
    111 * np.cos(np.radians(BASE_LAT_FOR_LON_SCALE))
)

VIS_LAT_GRID_SIZE = (VIS_GRID_NM * 1.852) / 111
VIS_LON_GRID_SIZE = (VIS_GRID_NM * 1.852) / (
    111 * np.cos(np.radians(BASE_LAT_FOR_LON_SCALE))
)

print("TRAIN_LAT_GRID_SIZE:", TRAIN_LAT_GRID_SIZE)
print("TRAIN_LON_GRID_SIZE:", TRAIN_LON_GRID_SIZE)
print("VIS_LAT_GRID_SIZE:", VIS_LAT_GRID_SIZE)
print("VIS_LON_GRID_SIZE:", VIS_LON_GRID_SIZE)

In [ ]:
# 6. AIS ??? 5? ?? resampling ??
# ????? ???? ???. ?? ??? ?? 5? bin? ???.
# COG/Heading? ?? ??? ????.

RESAMPLE_INTERVAL = "5min"
MAX_REASONABLE_SPEED_KNOTS = 40
MAX_REASONABLE_STEP_KM = 6


def circular_mean_deg(series):
    values = series.dropna().values

    if len(values) == 0:
        return np.nan

    radians = np.radians(values)
    sin_mean = np.mean(np.sin(radians))
    cos_mean = np.mean(np.cos(radians))

    angle = np.degrees(np.arctan2(sin_mean, cos_mean))
    return angle + 360 if angle < 0 else angle


def haversine_km_np(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c


def resample_ais_5min(df, progress_step=10):
    result = []
    grouped = list(df.groupby("MMSI"))
    total = len(grouped)
    next_progress = progress_step

    print(f"Resampling start: {total} vessels")

    for idx, (mmsi, group) in enumerate(grouped, start=1):
        group = group.sort_values("timestamp").set_index("timestamp")

        resampled = group[["lat", "lon", "sog"]].resample(RESAMPLE_INTERVAL).mean()
        resampled["cog"] = group["cog"].resample(RESAMPLE_INTERVAL).apply(circular_mean_deg)
        resampled["heading"] = group["heading"].resample(RESAMPLE_INTERVAL).apply(circular_mean_deg)

        resampled = resampled.dropna()
        resampled["MMSI"] = mmsi
        result.append(resampled.reset_index())

        progress = idx / total * 100 if total else 100
        if progress >= next_progress or idx == total:
            print(f"Resampling progress: {progress:.1f}% ({idx}/{total} vessels)")
            next_progress += progress_step

    if len(result) == 0:
        return pd.DataFrame(columns=["timestamp", "lat", "lon", "sog", "cog", "heading", "MMSI"])

    return pd.concat(result, ignore_index=True)


def remove_unrealistic_jumps(df, max_speed_knots=MAX_REASONABLE_SPEED_KNOTS, max_step_km=MAX_REASONABLE_STEP_KM):
    cleaned = []
    removed_count = 0

    for mmsi, group in df.groupby("MMSI"):
        group = group.sort_values("timestamp").copy()

        prev_lat = group["lat"].shift(1)
        prev_lon = group["lon"].shift(1)
        prev_time = group["timestamp"].shift(1)

        dt_hours = (group["timestamp"] - prev_time).dt.total_seconds() / 3600
        dist_km = haversine_km_np(prev_lat, prev_lon, group["lat"], group["lon"])
        speed_knots = (dist_km / 1.852) / dt_hours

        valid = (
            prev_time.isna()
            | (
                (dt_hours > 0)
                & (speed_knots <= max_speed_knots)
                & (dist_km <= max_step_km)
            )
        )

        removed_count += int((~valid).sum())
        cleaned.append(group.loc[valid])

    result = pd.concat(cleaned, ignore_index=True) if cleaned else df.iloc[0:0].copy()
    print(f"Removed unrealistic jump rows: {removed_count}")
    return result


def add_direction_features(df):
    result = df.copy()

    result["cog_sin"] = np.sin(np.radians(result["cog"]))
    result["cog_cos"] = np.cos(np.radians(result["cog"]))
    result["heading_sin"] = np.sin(np.radians(result["heading"]))
    result["heading_cos"] = np.cos(np.radians(result["heading"]))

    return result

In [ ]:
# 7. ? ?? Train / Validation / Test ?? ? resampling
# ?? ??? ?? ???? ? ???? ?? ??? ?? ? ???? split ? resampling??.
# ????? ???? ??, ??? AIS jump? ??? ? ?? sin/cos feature? ????.

train_month_raw = moving_df[moving_df["month"].isin([4, 5, 6])].copy()
val_month_raw = moving_df[moving_df["month"].isin([7])].copy()
test_month_raw = moving_df[moving_df["month"].isin([8])].copy()

train_raw = resample_ais_5min(train_month_raw, progress_step=5)
val_raw = resample_ais_5min(val_month_raw, progress_step=5)
test_raw = resample_ais_5min(test_month_raw, progress_step=5)

train_raw = add_direction_features(remove_unrealistic_jumps(train_raw))
val_raw = add_direction_features(remove_unrealistic_jumps(val_raw))
test_raw = add_direction_features(remove_unrealistic_jumps(test_raw))

for split_df in [train_raw, val_raw, test_raw]:
    split_df["month"] = split_df["timestamp"].dt.month

moving_resampled = pd.concat([train_raw, val_raw, test_raw], ignore_index=True)
moving_resampled = moving_resampled.sort_values(["MMSI", "timestamp"]).reset_index(drop=True)

print("train_raw:", train_raw.shape)
print("val_raw:", val_raw.shape)
print("test_raw:", test_raw.shape)
print("all moving_resampled:", moving_resampled.shape)

In [ ]:
# 8. ???
# ?? feature? lat/lon/sog? ?? sin/cos? ????.
# target? ?? ?? ??? ??? ?? ?? ?? ?? 5? delta_lat/delta_lon??.

feature_cols = [
    "lat", "lon", "sog",
    "cog_sin", "cog_cos",
    "heading_sin", "heading_cos"
]
target_cols = ["delta_lat", "delta_lon"]

x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()


def build_delta_target_frame(raw_df, interval_minutes=5):
    rows = []
    expected_delta = pd.Timedelta(minutes=interval_minutes)

    for _, group in raw_df.groupby("MMSI"):
        group = group.sort_values("timestamp").reset_index(drop=True)

        next_lat = group["lat"].shift(-1)
        next_lon = group["lon"].shift(-1)
        next_time = group["timestamp"].shift(-1)

        is_continuous = (next_time - group["timestamp"]) == expected_delta
        delta_df = pd.DataFrame({
            "delta_lat": next_lat - group["lat"],
            "delta_lon": next_lon - group["lon"]
        })

        rows.append(delta_df.loc[is_continuous].dropna())

    if len(rows) == 0:
        return pd.DataFrame(columns=target_cols)

    return pd.concat(rows, ignore_index=True)


train_delta_targets = build_delta_target_frame(train_raw)

x_scaler.fit(train_raw[feature_cols])
y_scaler.fit(train_delta_targets[target_cols])


def apply_scaling(raw_df):
    norm_df = raw_df.copy()
    norm_df[feature_cols] = x_scaler.transform(raw_df[feature_cols])
    return norm_df


train_norm = apply_scaling(train_raw)
val_norm = apply_scaling(val_raw)
test_norm = apply_scaling(test_raw)
all_norm = apply_scaling(moving_resampled)

print("delta target rows for scaler:", len(train_delta_targets))
train_norm.head()

In [ ]:
# 9. Interaction transition statistics ??
# train_raw?? grid ? ?? ??? ????.
# ?? sequence? interaction adjacency matrix? 10??? ? ??? ??? ???.


def latlon_to_grid(lat, lon, lat_min, lon_min, lat_grid_size, lon_grid_size):
    row = int((lat - lat_min) / lat_grid_size)
    col = int((lon - lon_min) / lon_grid_size)
    return row, col


def build_interaction_counts(df):
    transition_counts = defaultdict(int)

    lat_min = df["lat"].min()
    lon_min = df["lon"].min()

    for mmsi, group in df.groupby("MMSI"):
        group = group.sort_values("timestamp")
        coords = group[["lat", "lon"]].values

        for i in range(len(coords) - 1):
            g1 = latlon_to_grid(
                coords[i][0], coords[i][1],
                lat_min, lon_min,
                TRAIN_LAT_GRID_SIZE, TRAIN_LON_GRID_SIZE
            )
            g2 = latlon_to_grid(
                coords[i + 1][0], coords[i + 1][1],
                lat_min, lon_min,
                TRAIN_LAT_GRID_SIZE, TRAIN_LON_GRID_SIZE
            )
            transition_counts[(g1, g2)] += 1

    return transition_counts, lat_min, lon_min


transition_counts, train_lat_min, train_lon_min = build_interaction_counts(train_raw)

print("interaction transition count:", len(transition_counts))
print("train_lat_min:", train_lat_min)
print("train_lon_min:", train_lon_min)

In [ ]:
# 10. Sequence? Multi-Graph ?? ??
# Distance graph? sequence? ??? ?? ????,
# Interaction graph? 9??? ?? train ?? transition statistics? ????.

DISTANCE_GRAPH_SIGMA_KM = 0.5


def create_distance_graph(raw_seq, sigma_km=DISTANCE_GRAPH_SIGMA_KM):
    coords = raw_seq[:, :2]

    lat_km = coords[:, 0] * 111.0
    lon_km = coords[:, 1] * 111.0 * np.cos(np.radians(BASE_LAT_FOR_LON_SCALE))
    xy = np.stack([lat_km, lon_km], axis=1)

    diff = xy[:, None, :] - xy[None, :, :]
    dist_km = np.sqrt(np.sum(diff ** 2, axis=-1))

    A = np.exp(-dist_km / sigma_km)
    return A.astype(np.float32)


def create_interaction_graph(raw_seq):
    N = len(raw_seq)
    A = np.zeros((N, N), dtype=np.float32)

    nodes = [
        latlon_to_grid(
            row[0], row[1],
            train_lat_min, train_lon_min,
            TRAIN_LAT_GRID_SIZE, TRAIN_LON_GRID_SIZE
        )
        for row in raw_seq
    ]

    for i in range(N):
        for j in range(N):
            A[i, j] = transition_counts.get((nodes[i], nodes[j]), 0)

    if A.max() > 0:
        A = A / (A.max() + 1e-6)

    return A.astype(np.float32)


def create_multi_graph(raw_seq):
    A_distance = create_distance_graph(raw_seq)
    A_interaction = create_interaction_graph(raw_seq)
    return np.stack([A_distance, A_interaction], axis=0).astype(np.float32)

In [ ]:
# 11. ?? 10? ???? ?? 5? ??? ?? sequence ??
# 5? ??? ??? sequence? ????.


def create_next_step_sequences_with_graph(raw_df, norm_df, seq_len=10, stride=1, max_sequences=None):
    sequences = []
    expected_delta = np.timedelta64(5, "m")

    for mmsi in raw_df["MMSI"].unique():
        raw_group = raw_df[raw_df["MMSI"] == mmsi].sort_values("timestamp").reset_index(drop=True)
        norm_group = norm_df[norm_df["MMSI"] == mmsi].sort_values("timestamp").reset_index(drop=True)

        if len(raw_group) <= seq_len:
            continue

        raw_data = raw_group[feature_cols].values.astype(np.float32)
        norm_data = norm_group[feature_cols].values.astype(np.float32)
        timestamps = raw_group["timestamp"].values.astype("datetime64[ns]")

        for i in range(0, len(raw_group) - seq_len, stride):
            time_window = timestamps[i:i + seq_len + 1]
            time_diffs = np.diff(time_window).astype("timedelta64[m]")

            if not np.all(time_diffs == expected_delta):
                continue

            raw_seq = raw_data[i:i + seq_len]
            norm_seq = norm_data[i:i + seq_len]

            current_pos = raw_data[i + seq_len - 1][:2]
            next_pos = raw_data[i + seq_len][:2]
            raw_delta = next_pos - current_pos

            target = y_scaler.transform(
                pd.DataFrame([raw_delta], columns=target_cols)
            )[0].astype(np.float32)

            graph = create_multi_graph(raw_seq)
            sequences.append((norm_seq, graph, target))

            if max_sequences is not None and len(sequences) >= max_sequences:
                return sequences

    return sequences


SEQ_LEN = 10
TRAIN_STRIDE = 1
VAL_STRIDE = 1
TEST_STRIDE = 1

MAX_TRAIN_SEQUENCES = None
MAX_VAL_SEQUENCES = None
MAX_TEST_SEQUENCES = None

train_sequences = create_next_step_sequences_with_graph(
    train_raw, train_norm, seq_len=SEQ_LEN, stride=TRAIN_STRIDE, max_sequences=MAX_TRAIN_SEQUENCES
)
val_sequences = create_next_step_sequences_with_graph(
    val_raw, val_norm, seq_len=SEQ_LEN, stride=VAL_STRIDE, max_sequences=MAX_VAL_SEQUENCES
)
test_sequences = create_next_step_sequences_with_graph(
    test_raw, test_norm, seq_len=SEQ_LEN, stride=TEST_STRIDE, max_sequences=MAX_TEST_SEQUENCES
)

print("train_sequences:", len(train_sequences))
print("val_sequences:", len(val_sequences))
print("test_sequences:", len(test_sequences))
print("sample norm_seq:", train_sequences[0][0].shape)
print("sample graph:", train_sequences[0][1].shape)
print("sample target:", train_sequences[0][2].shape)

In [ ]:
# 12. Dataset / DataLoader
# graph는 이미 pre-compute 되어 있음

class AISDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        norm_seq, graph, target = self.sequences[idx]

        return (
            torch.tensor(norm_seq, dtype=torch.float32),
            torch.tensor(graph, dtype=torch.float32),
            torch.tensor(target, dtype=torch.float32)
        )


BATCH_SIZE = 32

train_dataset = AISDataset(train_sequences)
val_dataset = AISDataset(val_sequences)
test_dataset = AISDataset(test_sequences)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

for X, graphs, y in train_loader:
    print("X:", X.shape)
    print("graphs:", graphs.shape)
    print("y:", y.shape)
    break

In [ ]:
# 13. STMGCN ?? ??
# Spatial-Temporal Multi-Graph Convolutional Network

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.1):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, X, A):
        B, T, _ = X.shape

        I = torch.eye(T, device=X.device, dtype=X.dtype).unsqueeze(0).expand(B, T, T)
        A_hat = A.to(dtype=X.dtype) + I

        degree = torch.sum(A_hat, dim=-1).clamp_min(1e-6)
        degree_inv_sqrt = torch.pow(degree, -0.5)
        A_norm = degree_inv_sqrt.unsqueeze(-1) * A_hat * degree_inv_sqrt.unsqueeze(-2)

        H = A_norm @ self.linear(X)
        H = F.relu(H)
        H = self.dropout(H)

        return self.norm(H)


class MultiGraphAttentionFusion(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.score_layer = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, H_list):
        scores = []

        for H in H_list:
            graph_score = self.score_layer(H).mean(dim=1)
            scores.append(graph_score)

        scores = torch.cat(scores, dim=1)
        weights = F.softmax(scores, dim=1)

        fused = torch.zeros_like(H_list[0])
        for i, H in enumerate(H_list):
            w = weights[:, i].view(-1, 1, 1)
            fused = fused + w * H

        return fused, weights


class TemporalConvBlock(nn.Module):
    def __init__(self, hidden_dim, kernel_size=3, dropout=0.1):
        super().__init__()
        padding = kernel_size // 2

        self.filter_conv = nn.Conv1d(hidden_dim, hidden_dim, kernel_size, padding=padding)
        self.gate_conv = nn.Conv1d(hidden_dim, hidden_dim, kernel_size, padding=padding)
        self.proj = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=1)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        residual = x

        x = x.transpose(1, 2)
        z = torch.tanh(self.filter_conv(x))
        g = torch.sigmoid(self.gate_conv(x))
        x = self.proj(z * g)
        x = self.dropout(x)
        x = x.transpose(1, 2)

        return self.norm(x + residual)


class STMGCN(nn.Module):
    def __init__(
        self,
        input_dim=5,
        hidden_dim=64,
        num_graphs=2,
        output_dim=2,
        num_temporal_blocks=2,
        dropout=0.1
    ):
        super().__init__()

        self.num_graphs = num_graphs

        self.input_projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim)
        )

        self.gcn_layers = nn.ModuleList([
            GCNLayer(hidden_dim, hidden_dim, dropout=dropout)
            for _ in range(num_graphs)
        ])

        self.graph_fusion = MultiGraphAttentionFusion(hidden_dim)

        self.temporal_blocks = nn.ModuleList([
            TemporalConvBlock(hidden_dim, dropout=dropout)
            for _ in range(num_temporal_blocks)
        ])

        self.output_layer = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, output_dim)
        )

        self.last_graph_weights = None

    def forward(self, X, graphs):
        H0 = self.input_projection(X)

        H_list = []
        for i in range(self.num_graphs):
            A = graphs[:, i, :, :]
            H = self.gcn_layers[i](H0, A)
            H_list.append(H)

        H, graph_weights = self.graph_fusion(H_list)
        self.last_graph_weights = graph_weights.detach()

        for block in self.temporal_blocks:
            H = block(H)

        last_hidden = H[:, -1, :]
        return self.output_layer(last_hidden)

In [ ]:
# 14. ?? ??

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

model = STMGCN(
    input_dim=len(feature_cols),
    hidden_dim=64,
    num_graphs=2,
    output_dim=2,
    num_temporal_blocks=2,
    dropout=0.1
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
criterion = nn.MSELoss()

In [ ]:
# 15. ?? ?? + Validation ?? + Early Stopping


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for X, graphs, y in loader:
        X = X.to(device)
        graphs = graphs.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        pred = model(X, graphs)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate_loss(model, loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for X, graphs, y in loader:
            X = X.to(device)
            graphs = graphs.to(device)
            y = y.to(device)

            pred = model(X, graphs)
            loss = criterion(pred, y)
            total_loss += loss.item()

    return total_loss / len(loader)


EPOCHS = 2000
PATIENCE = 150

best_val_loss = float("inf")
patience_counter = 0
best_model_state = None

train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss = evaluate_loss(model, val_loader, criterion)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch [{epoch + 1}/{EPOCHS}] Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print("Early stopping")
        break

if best_model_state is not None:
    model.load_state_dict(best_model_state)

print("Best Val Loss:", best_val_loss)

In [ ]:
# 16. 학습 곡선 확인

plt.figure(figsize=(8, 5))

plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training / Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 17. Test set ?? 5? ??? ?? ??

model.eval()

with torch.no_grad():
    for X, graphs, y in test_loader:
        X = X.to(device)
        graphs = graphs.to(device)
        y = y.to(device)

        pred = model(X, graphs)

        pred_delta = y_scaler.inverse_transform(pred.cpu().numpy())
        true_delta = y_scaler.inverse_transform(y.cpu().numpy())

        print("?? delta_lat/delta_lon:")
        print(pred_delta[:5])

        print("?? delta_lat/delta_lon:")
        print(true_delta[:5])

        break

In [ ]:
# 18. ?? ?? ?? ??
# 5? one-step STMGCN? ?? ??? multi-horizon trajectory? ????.

FORECAST_MINUTES_LIST = [5, 10, 15, 30, 60, 120, 180, 240]
MAX_FORECAST_MINUTES = max(FORECAST_MINUTES_LIST)
TOLERANCE_MINUTES = 2.5


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c


def estimate_sog_cog_from_points(prev_lat, prev_lon, new_lat, new_lon, minutes=5):
    dist_km = haversine_km(prev_lat, prev_lon, new_lat, new_lon)
    sog_knots = (dist_km / 1.852) / (minutes / 60)

    dlon = np.radians(new_lon - prev_lon)
    lat1 = np.radians(prev_lat)
    lat2 = np.radians(new_lat)

    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    cog = (np.degrees(np.arctan2(y, x)) + 360) % 360

    return sog_knots, cog


def predict_next_position(model, raw_seq, norm_seq):
    graph = create_multi_graph(raw_seq)

    X = torch.tensor(norm_seq, dtype=torch.float32).unsqueeze(0).to(device)
    graphs = torch.tensor(graph, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        pred_delta_norm = model(X, graphs).cpu().numpy()

    pred_delta = y_scaler.inverse_transform(pred_delta_norm)[0]
    last_latlon = raw_seq[-1, :2]
    pred_latlon = last_latlon + pred_delta

    return pred_delta_norm[0], pred_delta, pred_latlon


def recursive_forecast_path(model, raw_seq, norm_seq, forecast_minutes=240):
    steps = int(forecast_minutes / 5)

    raw_seq = raw_seq.copy()
    norm_seq = norm_seq.copy()
    predicted_path = []

    for step in range(1, steps + 1):
        pred_delta_norm, pred_delta, pred_latlon = predict_next_position(model, raw_seq, norm_seq)

        prev_raw = raw_seq[-1].copy()
        new_raw = prev_raw.copy()
        new_raw[0] = pred_latlon[0]
        new_raw[1] = pred_latlon[1]

        est_sog, est_cog = estimate_sog_cog_from_points(
            prev_raw[0], prev_raw[1],
            pred_latlon[0], pred_latlon[1],
            minutes=5
        )

        new_raw[2] = est_sog
        new_raw[3] = np.sin(np.radians(est_cog))
        new_raw[4] = np.cos(np.radians(est_cog))
        new_raw[5] = np.sin(np.radians(est_cog))
        new_raw[6] = np.cos(np.radians(est_cog))

        new_norm = x_scaler.transform(pd.DataFrame([new_raw], columns=feature_cols))[0]

        raw_seq = np.vstack([raw_seq[1:], new_raw])
        norm_seq = np.vstack([norm_seq[1:], new_norm])

        predicted_path.append({
            "step": step,
            "forecast_minutes": step * 5,
            "lat": pred_latlon[0],
            "lon": pred_latlon[1]
        })

    return pd.DataFrame(predicted_path)


def generate_base_times(df, forecast_minutes, num_base_times=10):
    all_times = sorted(df["timestamp"].unique())
    max_allowed_time = df["timestamp"].max() - pd.Timedelta(minutes=forecast_minutes)
    candidate_times = [t for t in all_times if t <= max_allowed_time]

    if len(candidate_times) == 0:
        raise ValueError("?? ?? ??? ??? ? ?? base_time? ????.")

    if len(candidate_times) <= num_base_times:
        return candidate_times

    selected_idx = np.linspace(0, len(candidate_times) - 1, num_base_times).astype(int)
    return [candidate_times[i] for i in selected_idx]


def make_prediction_inputs_at_base_time(df_raw, df_norm, base_time, seq_len=10):
    pred_inputs = []

    current_df = df_raw[df_raw["timestamp"] <= base_time].copy()
    if len(current_df) == 0:
        return pred_inputs

    latest_idx = current_df.groupby("MMSI")["timestamp"].idxmax()
    current_snapshot = current_df.loc[latest_idx].copy()
    moving_snapshot = current_snapshot[current_snapshot["sog"] > MOVING_THRESHOLD].copy()

    for _, row in moving_snapshot.iterrows():
        mmsi = row["MMSI"]
        current_time = row["timestamp"]

        hist_raw = df_raw[
            (df_raw["MMSI"] == mmsi) &
            (df_raw["timestamp"] <= current_time)
        ].sort_values("timestamp").tail(seq_len)

        hist_norm = df_norm[
            (df_norm["MMSI"] == mmsi) &
            (df_norm["timestamp"] <= current_time)
        ].sort_values("timestamp").tail(seq_len)

        if len(hist_raw) < seq_len or len(hist_norm) < seq_len:
            continue

        time_diffs = hist_raw["timestamp"].diff().dropna()
        if not (time_diffs == pd.Timedelta(minutes=5)).all():
            continue

        pred_inputs.append({
            "MMSI": mmsi,
            "base_time": current_time,
            "start_lat": hist_raw.iloc[-1]["lat"],
            "start_lon": hist_raw.iloc[-1]["lon"],
            "raw_seq": hist_raw[feature_cols].values.astype(np.float32),
            "norm_seq": hist_norm[feature_cols].values.astype(np.float32)
        })

    return pred_inputs


def get_actual_position_near_time(df, mmsi, target_time, tolerance_minutes=2.5):
    tolerance = pd.Timedelta(minutes=tolerance_minutes)
    vessel_df = df[
        (df["MMSI"] == mmsi) &
        (df["timestamp"] >= target_time - tolerance) &
        (df["timestamp"] <= target_time + tolerance)
    ].copy()

    if len(vessel_df) == 0:
        return None

    vessel_df["time_diff"] = (vessel_df["timestamp"] - target_time).abs()
    row = vessel_df.sort_values("time_diff").iloc[0]

    return {
        "timestamp": row["timestamp"],
        "lat": row["lat"],
        "lon": row["lon"]
    }


def get_actual_path_for_horizons(df, mmsi, base_time, forecast_minutes_list):
    rows = []

    for minutes in forecast_minutes_list:
        actual = get_actual_position_near_time(
            df,
            mmsi,
            base_time + pd.Timedelta(minutes=minutes),
            tolerance_minutes=TOLERANCE_MINUTES
        )

        if actual is None:
            continue

        rows.append({
            "forecast_minutes": minutes,
            "timestamp": actual["timestamp"],
            "lat": actual["lat"],
            "lon": actual["lon"]
        })

    return pd.DataFrame(rows)

In [ ]:
# 19. Multi-horizon ?? ?? ?? ??
# ? ??? ?? 4?? path? ? ?? ????, ??? horizon ??? ????.

NUM_BASE_TIMES = 10

base_times = generate_base_times(
    test_raw,
    forecast_minutes=MAX_FORECAST_MINUTES,
    num_base_times=NUM_BASE_TIMES
)

trajectory_records = []
trajectory_paths = []

model.eval()

for base_idx, base_time in enumerate(base_times):
    print(f"[{base_idx + 1}/{len(base_times)}] base_time: {base_time}")

    pred_inputs = make_prediction_inputs_at_base_time(
        test_raw,
        test_norm,
        base_time,
        seq_len=SEQ_LEN
    )

    if len(pred_inputs) == 0:
        print("?? ??? ?? ?? ??. skip")
        continue

    for item in pred_inputs:
        mmsi = item["MMSI"]
        vessel_base_time = item["base_time"]

        pred_path = recursive_forecast_path(
            model,
            item["raw_seq"],
            item["norm_seq"],
            forecast_minutes=MAX_FORECAST_MINUTES
        )

        actual_path = get_actual_path_for_horizons(
            test_raw,
            mmsi,
            vessel_base_time,
            FORECAST_MINUTES_LIST
        )

        if len(actual_path) == 0:
            continue

        pred_horizon = pred_path[pred_path["forecast_minutes"].isin(FORECAST_MINUTES_LIST)].copy()

        merged = pred_horizon.merge(
            actual_path,
            on="forecast_minutes",
            suffixes=("_pred", "_actual")
        )

        for _, row in merged.iterrows():
            error_km = haversine_km(
                row["lat_actual"], row["lon_actual"],
                row["lat_pred"], row["lon_pred"]
            )

            trajectory_records.append({
                "base_time": vessel_base_time,
                "MMSI": mmsi,
                "forecast_minutes": row["forecast_minutes"],
                "pred_lat": row["lat_pred"],
                "pred_lon": row["lon_pred"],
                "actual_lat": row["lat_actual"],
                "actual_lon": row["lon_actual"],
                "error_km": error_km,
                "error_nm": error_km / 1.852
            })

        trajectory_paths.append({
            "base_time": vessel_base_time,
            "MMSI": mmsi,
            "start_lat": item["start_lat"],
            "start_lon": item["start_lon"],
            "pred_path": pred_horizon.reset_index(drop=True),
            "actual_path": actual_path.reset_index(drop=True)
        })

trajectory_results_df = pd.DataFrame(trajectory_records)
print("trajectory result rows:", len(trajectory_results_df))
print("trajectory paths:", len(trajectory_paths))
trajectory_results_df.head()

In [ ]:
# 20. ?? ??? ??/?? 10? ?? ??
# ??: 5? horizon? error_km

TOP_N = 10
SELECTION_HORIZON = 5
PLOT_HORIZONS = [5, 10, 15, 30, 60]


def select_paths_by_error(trajectory_results_df, trajectory_paths, top=True, n=10, horizon=5):
    selected_df = (
        trajectory_results_df[trajectory_results_df["forecast_minutes"] == horizon]
        .sort_values("error_km", ascending=top)
        .drop_duplicates(subset=["MMSI"])
        .head(n)
        .reset_index(drop=True)
    )

    selected_paths = []

    for _, row in selected_df.iterrows():
        matched = [
            item for item in trajectory_paths
            if item["MMSI"] == row["MMSI"] and item["base_time"] == row["base_time"]
        ]

        if len(matched) > 0:
            selected_paths.append(matched[0])

    return selected_df, selected_paths


def get_plot_nodes_5min(item, plot_horizons):
    pred_nodes = item["pred_path"][item["pred_path"]["forecast_minutes"].isin(plot_horizons)].copy()
    actual_nodes = item["actual_path"][item["actual_path"]["forecast_minutes"].isin(plot_horizons)].copy()

    merged = pred_nodes.merge(
        actual_nodes,
        on="forecast_minutes",
        suffixes=("_pred", "_actual")
    )

    merged = merged.sort_values("forecast_minutes").reset_index(drop=True)

    if len(merged) == 0:
        return None

    return merged


top10_df, top10_paths = select_paths_by_error(
    trajectory_results_df, trajectory_paths, top=True, n=TOP_N, horizon=SELECTION_HORIZON
)
bottom10_df, bottom10_paths = select_paths_by_error(
    trajectory_results_df, trajectory_paths, top=False, n=TOP_N, horizon=SELECTION_HORIZON
)

print("?? ??? ?? 10? ??")
display(top10_df[["MMSI", "base_time", "forecast_minutes", "error_km", "error_nm"]])

print("?? ??? ?? 10? ??")
display(bottom10_df[["MMSI", "base_time", "forecast_minutes", "error_km", "error_nm"]])

In [ ]:
# 21. ??/?? 10? ??: ?? ??? vs ?? ??? ???
# ??: 5?, 10?, 15?, 30?, 1??


def plot_pred_actual_paths(selected_paths, title):
    plt.figure(figsize=(11, 9))
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(selected_paths), 1)))
    plot_count = 0

    for i, item in enumerate(selected_paths):
        color = colors[i]
        merged = get_plot_nodes_5min(item, PLOT_HORIZONS)

        if merged is None:
            continue

        plot_count += 1

        plt.scatter(
            item["start_lon"], item["start_lat"],
            color=color, marker="o", s=70,
            edgecolor="black", linewidth=0.8
        )

        plt.plot(
            merged["lon_actual"], merged["lat_actual"],
            color=color, linestyle="-", marker="x", linewidth=1.5,
            label=f"Actual {i}: {item['MMSI']}"
        )

        plt.plot(
            merged["lon_pred"], merged["lat_pred"],
            color=color, linestyle="--", marker="*", linewidth=1.5,
            label=f"Pred {i}: {item['MMSI']}"
        )

        for _, row in merged.iterrows():
            plt.text(
                row["lon_pred"], row["lat_pred"],
                f"{int(row['forecast_minutes'])}m",
                fontsize=7,
                color=color
            )

    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title(title)
    plt.grid(True)
    plt.legend(fontsize=8, ncol=2)

    ax = plt.gca()
    ax.xaxis.set_major_formatter(ScalarFormatter(useOffset=False))
    ax.yaxis.set_major_formatter(ScalarFormatter(useOffset=False))
    ax.ticklabel_format(style="plain", axis="both")

    plt.show()
    print("plotted vessels:", plot_count)


plot_pred_actual_paths(
    top10_paths,
    "Top 10 Vessels: Predicted vs Actual Trajectories"
)

plot_pred_actual_paths(
    bottom10_paths,
    "Bottom 10 Vessels: Predicted vs Actual Trajectories"
)

In [ ]:
# 22. ??/?? 10? ??: ??-?? ??? ???
# ?? horizon? ?? ??? ?? ??? ??? ????.


def plot_error_line_paths(selected_paths, title):
    plt.figure(figsize=(11, 9))
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(selected_paths), 1)))
    plot_count = 0

    for i, item in enumerate(selected_paths):
        color = colors[i]
        merged = get_plot_nodes_5min(item, PLOT_HORIZONS)

        if merged is None:
            continue

        plot_count += 1

        plt.scatter(
            item["start_lon"], item["start_lat"],
            color=color, marker="o", s=70,
            edgecolor="black", linewidth=0.8,
            label=f"Start {i}: {item['MMSI']}"
        )

        plt.plot(
            merged["lon_actual"], merged["lat_actual"],
            color=color, linestyle="-", marker="x", linewidth=1.5,
            label=f"Actual {i}: {item['MMSI']}"
        )

        plt.plot(
            merged["lon_pred"], merged["lat_pred"],
            color=color, linestyle="--", marker="*", linewidth=1.5,
            label=f"Pred {i}: {item['MMSI']}"
        )

        for _, row in merged.iterrows():
            plt.plot(
                [row["lon_actual"], row["lon_pred"]],
                [row["lat_actual"], row["lat_pred"]],
                color=color,
                alpha=0.35,
                linewidth=0.9
            )

            mid_lon = (row["lon_actual"] + row["lon_pred"]) / 2
            mid_lat = (row["lat_actual"] + row["lat_pred"]) / 2
            plt.text(mid_lon, mid_lat, f"{int(row['forecast_minutes'])}m", fontsize=7, color=color)

    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title(title)
    plt.grid(True)
    plt.legend(fontsize=8, ncol=2)

    ax = plt.gca()
    ax.xaxis.set_major_formatter(ScalarFormatter(useOffset=False))
    ax.yaxis.set_major_formatter(ScalarFormatter(useOffset=False))
    ax.ticklabel_format(style="plain", axis="both")

    plt.show()
    print("plotted vessels:", plot_count)


plot_error_line_paths(
    top10_paths,
    "Top 10 Vessels: Error Lines Between Actual and Predicted Positions"
)

plot_error_line_paths(
    bottom10_paths,
    "Bottom 10 Vessels: Error Lines Between Actual and Predicted Positions"
)

In [ ]:
# 23. ?? ??: Horizon? ?? ?? ?? ??
# ?? ??? ??? ? ???? ????.

if len(trajectory_results_df) == 0:
    raise ValueError("trajectory_results_df? ?? ????. 19? ?? ?? ?? ?? ?????.")

trajectory_summary_df = (
    trajectory_results_df
    .groupby("forecast_minutes")
    .agg(
        valid_count=("error_km", "count"),
        mean_error_km=("error_km", "mean"),
        median_error_km=("error_km", "median"),
        rmse_km=("error_km", lambda x: np.sqrt(np.mean(np.square(x)))),
        p90_error_km=("error_km", lambda x: np.percentile(x, 90)),
        max_error_km=("error_km", "max"),
        mean_error_nm=("error_nm", "mean"),
        median_error_nm=("error_nm", "median")
    )
    .reset_index()
)

display(trajectory_summary_df)

plt.figure(figsize=(10, 5))
plt.plot(
    trajectory_summary_df["forecast_minutes"],
    trajectory_summary_df["mean_error_km"],
    marker="o",
    label="Mean error"
)
plt.plot(
    trajectory_summary_df["forecast_minutes"],
    trajectory_summary_df["median_error_km"],
    marker="s",
    label="Median error"
)
plt.plot(
    trajectory_summary_df["forecast_minutes"],
    trajectory_summary_df["p90_error_km"],
    marker="^",
    label="P90 error"
)

plt.xlabel("Forecast Horizon (minutes)")
plt.ylabel("Haversine Error (km)")
plt.title("Trajectory Prediction Error by Forecast Horizon")
plt.grid(True)
plt.legend()
plt.show()